<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 08. Bahdanau Additive Attention — Neural Machine Translation by Jointly Learning to Align and Translate

- **원 논문:** [Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473)
- **저자 / 발표:** Dzmitry Bahdanau, Kyunghyun Cho, Yoshua Bengio · ICLR 2015 (2015)
- **난이도 / 예상 시간:** 중급 · 약 65분
- **선수 지식:** encoder-decoder, softmax, masking, batched matrix multiplication

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

decoder query와 모든 encoder annotation 사이 additive energy를 계산하고, padding을 softmax 전에 가린 뒤 정렬 확률의 가중합으로 context를 만든다. 작은 단조 정렬 과제에서 attention weight가 올바른 위치를 찾는지 측정한다.

**원 논문과 다른 것**

원 논문은 양방향 RNN encoder와 gated decoder를 WMT'14 영어-프랑스어 번역에 학습해 BLEU와 alignment를 평가했다. 여기서는 이미 만들어진 6차원 annotation과 5개 query만 사용하며 번역기 전체나 BLEU를 재현하지 않는다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §3.1, 식 (6): context vector는 annotation의 가중합 | `AdditiveAttention.forward`의 `context = bmm(alpha, annotations)` | context shape와 attention 행 합 1을 assert |
| §3.1, 식 (7): energy를 softmax로 정규화해 α_ij 계산 | `AdditiveAttention.forward`의 masked energy와 `softmax(dim=-1)` | padding 열의 확률이 정확히 0인지 검사 |
| §3.1, 식 (8) 및 Appendix A.1.2: feed-forward alignment model a(s,h) | `AdditiveAttention.score`의 `v(tanh(W_s s + W_h h))` | energy shape 및 모든 파라미터 gradient 확인 |
| Figure 3: target 위치와 source annotation 사이 soft alignment | 합성 monotonic alignment 정확도와 attention heatmap | argmax alignment accuracy 및 padding mass로 정량 검증 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
e_{t,i}=v_a^\top\tanh(W_s s_{t-1}+W_h h_i),\quad
\alpha_{t,i}=\operatorname{softmax}_i(e_{t,i}),\quad
c_t=\sum_i\alpha_{t,i}h_i
$$

- $h_i$는 encoder state, $s_{t-1}$는 decoder state, $\alpha_{t,i}$는 alignment weight입니다.
- 하나의 고정 context 대신 decoder step마다 source의 다른 위치를 가중 합산합니다.
- energy, masked softmax, context aggregation을 분리해 alignment을 직접 검증합니다.
- score와 weight는 $[B,T,S]$, context는 $[B,T,H]$이며 padding 위치 가중치는 0이어야 합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §3.1, 식 (6): context vector는 annotation의 가중합
- **노트북 구현:** `AdditiveAttention.forward`의 `context = bmm(alpha, annotations)`
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** context shape와 attention 행 합 1을 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문은 양방향 RNN encoder와 gated decoder를 WMT'14 영어-프랑스어 번역에 학습해 BLEU와 alignment를 평가했다. 여기서는 이미 만들어진 6차원 annotation과 5개 query만 사용하며 번역기 전체나 BLEU를 재현하지 않는다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 핵심 아이디어

하나의 고정 길이 벡터 대신 decoder step $i$마다
$e_{ij}=v_a^\top\tanh(W_a s_{i-1}+U_a h_j)$를 계산한다. 이를 source 축으로
softmax한 $\alpha_{ij}$가 soft alignment이고,
$c_i=\sum_j\alpha_{ij}h_j$가 현재 step의 context다(식 (6)--(8)).

논문에는 padding mask가 수식으로 명시되지 않지만, minibatch 구현에서는 가짜
annotation이 확률 질량을 받지 않도록 energy를 softmax **전에** 가려야 한다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** encoder/decoder [B,S/T,H] → attention [B,T,S] → context [B,T,H]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,torch.ones,torch.bool,torch.arange,ACCELERATOR.move -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 5개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

<details>
<summary><code>torch.bool</code></summary>

#### `torch.bool`

- **역할:** 불리언 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** mask와 조건 결과를 표현합니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.bool](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

SEED = 808
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
print(ACCELERATOR.summary())

# 다섯 decoder step, 여섯 encoder slot(마지막은 padding)
STEPS, SOURCE_SLOTS, STATE_DIM = 5, 6, 6
annotations = torch.eye(SOURCE_SLOTS).unsqueeze(0).expand(STEPS, -1, -1).clone()
decoder_queries = torch.eye(STATE_DIM)[:STEPS]
source_mask = torch.ones((STEPS, SOURCE_SLOTS), dtype=torch.bool)
source_mask[:, -1] = False
alignment_targets = torch.arange(STEPS)
annotations, decoder_queries, source_mask, alignment_targets = ACCELERATOR.move(
    annotations, decoder_queries, source_mask, alignment_targets
)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3.1, 식 (6): context vector는 annotation의 가중합 / §3.1, 식 (7): energy를 softmax로 정규화해 α_ij 계산 / §3.1, 식 (8) 및 Appendix A.1.2: feed-forward alignment model a(s,h)
- **이 셀의 책임:** `AdditiveAttention.forward`의 `context = bmm(alpha, annotations)` / `AdditiveAttention.forward`의 masked energy와 `softmax(dim=-1)` / `AdditiveAttention.score`의 `v(tanh(W_s s + W_h h))`
- **Tensor shape 계약:** encoder/decoder [B,S/T,H] → attention [B,T,S] → context [B,T,H]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** context shape와 attention 행 합 1을 assert / padding 열의 확률이 정확히 0인지 검사 / energy shape 및 모든 파라미터 gradient 확인. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Linear,torch.tanh,Tensor.masked_fill,torch.softmax,torch.bmm,torch.allclose,Tensor.sum -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 7개 · 수식 7개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>torch.tanh(input)</code></summary>

#### `torch.tanh(input)`

- **역할:** 실수를 -1과 1 사이 값으로 바꿉니다.
- **입력·반환:** 실수 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** RNN hidden state나 범위가 제한된 출력을 만듭니다.
- **주의점:** 큰 절댓값에서 포화되어 gradient가 작아질 수 있습니다.

##### 관련 수식과 코드 연결

$$
\tanh(x)=\frac{e^x-e^{-x}}{e^x+e^{-x}}
$$

- **기호:** $x$는 실수이고 출력은 -1과 1 사이입니다.
- **수식 → 코드:** API가 activation을 0 중심의 제한된 범위로 압축합니다.
- **직관:** 부호는 유지하면서 큰 값의 영향을 제한합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [torch.tanh](https://docs.pytorch.org/docs/stable/generated/torch.tanh.html)

</details>

<details>
<summary><code>tensor.masked_fill(mask, value)</code></summary>

#### `tensor.masked_fill(mask, value)`

- **역할:** mask가 참인 위치를 지정 값으로 바꿉니다.
- **입력·반환:** 불리언 mask와 값을 받고 새 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** padding이나 미래 token의 attention score를 선택에서 제외합니다.
- **주의점:** mask는 입력에 broadcast 가능해야 하고 softmax 전 값은 충분히 작아야 합니다.

##### 관련 수식과 코드 연결

$$
s'_{ij}=\begin{cases}s_{ij},&M_{ij}=0\\-\infty,&M_{ij}=1\end{cases}\quad\Rightarrow\quad\operatorname{softmax}(s')_{ij}=0\;\text{if }M_{ij}=1
$$

- **기호:** $s$는 score, $M$은 제외할 위치가 참인 mask입니다.
- **수식 → 코드:** attention에서는 mask 위치를 매우 작은 값으로 채운 뒤 softmax를 적용합니다.
- **직관:** 확률을 나중에 지우는 대신 정규화 분모에도 들어가지 못하게 합니다.
- **shape 확인:** mask는 score에 broadcast 가능해야 하며 보통 `[B, H, T_q, T_k]`를 사용합니다.
- **공식 문서:** [Tensor.masked_fill](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.masked_fill.html)

</details>

<details>
<summary><code>torch.softmax(input, dim, dtype=None)</code></summary>

#### `torch.softmax(input, dim, dtype=None)`

- **역할:** 지정 축의 값을 합이 1인 확률형 가중치로 변환합니다.
- **입력·반환:** 텐서와 축을 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** attention score나 class logit을 정규화합니다.
- **주의점:** `dim`을 잘못 고르면 엉뚱한 축이 정규화되고 큰 logit은 포화될 수 있습니다.

##### 관련 수식과 코드 연결

$$
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}},\qquad \log p_i=z_i-\log\sum_j e^{z_j}
$$

- **기호:** $z_i$는 logit, $p_i$는 선택한 축에서 합이 1인 확률입니다.
- **수식 → 코드:** 코드의 `dim`이 분모의 합을 계산할 $j$축을 결정합니다.
- **직관:** 각 logit을 다른 후보와 비교 가능한 상대 확률로 바꿉니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택한 축의 softmax 값 합은 1입니다.
- **공식 문서:** [torch.softmax](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html)

</details>

<details>
<summary><code>torch.bmm(input, mat2)</code></summary>

#### `torch.bmm(input, mat2)`

- **역할:** 3차원 batch 행렬곱을 수행합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
C_{ij}=\sum_{k=1}^{K}A_{ik}B_{kj}
$$

- **기호:** $k$는 곱한 뒤 사라지는 축이고 $i,j$는 결과에 남는 축입니다.
- **수식 → 코드:** `matmul`은 마지막 두 축에 이 합을 적용하고, `bmm`은 batch별로, `einsum`은 문자열로 축 관계를 지정합니다.
- **직관:** 공통 축 $k$의 정보를 가중합해 두 표현 사이의 상호작용을 만듭니다.
- **shape 확인:** `[..., I, K] @ [..., K, J] -> [..., I, J]`이며 앞쪽 batch 축은 broadcast될 수 있습니다.
- **공식 문서:** [torch.bmm](https://docs.pytorch.org/docs/stable/generated/torch.bmm.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class AdditiveAttention(nn.Module):
    def __init__(self, query_dim, annotation_dim, attention_dim):
        super().__init__()
        self.query_projection = nn.Linear(query_dim, attention_dim, bias=False)
        self.annotation_projection = nn.Linear(
            annotation_dim,
            attention_dim,
            bias=False,
        )
        self.energy_projection = nn.Linear(attention_dim, 1, bias=False)

    def score(self, query, encoder_annotations):
        if query.ndim != 2 or encoder_annotations.ndim != 3:
            raise ValueError("expected query [B,D] and annotations [B,S,D]")
        query_term = self.query_projection(query).unsqueeze(1)
        annotation_term = self.annotation_projection(encoder_annotations)
        joint_state = torch.tanh(query_term + annotation_term)
        return self.energy_projection(joint_state).squeeze(-1)

    def normalize(self, energy, valid_mask):
        if energy.shape != valid_mask.shape:
            raise ValueError("energy and valid_mask shapes must match")
        if not valid_mask.any(dim=1).all():
            raise ValueError("every sequence needs at least one valid annotation")
        masked_energy = energy.masked_fill(
            ~valid_mask,
            torch.finfo(energy.dtype).min,
        )
        alpha = torch.softmax(masked_energy, dim=-1)
        return alpha, masked_energy

    def compose_context(self, alpha, encoder_annotations):
        weighted = torch.bmm(alpha.unsqueeze(1), encoder_annotations)
        return weighted.squeeze(1)

    def forward(self, query, encoder_annotations, valid_mask):
        energy = self.score(query, encoder_annotations)
        alpha, masked_energy = self.normalize(energy, valid_mask)
        context = self.compose_context(alpha, encoder_annotations)
        return context, alpha, masked_energy


torch.manual_seed(SEED)
attention = AdditiveAttention(STATE_DIM, STATE_DIM, attention_dim=12).to(DEVICE)
context, alpha, energy = attention(decoder_queries, annotations, source_mask)
assert context.shape == (STEPS, STATE_DIM)
assert alpha.shape == (STEPS, SOURCE_SLOTS)
assert torch.allclose(alpha.sum(dim=1), alpha.new_ones(STEPS), atol=1e-6)
assert torch.all(alpha[:, -1] == 0)

### 구현 단계 3 · 목적함수·학습 update

- **원문 위치:** §3.1, 식 (6): context vector는 annotation의 가중합 / §3.1, 식 (7): energy를 softmax로 정규화해 α_ij 계산 / §3.1, 식 (8) 및 Appendix A.1.2: feed-forward alignment model a(s,h) / Figure 3: target 위치와 source annotation 사이 soft alignment
- **이 셀의 책임:** `AdditiveAttention.forward`의 `context = bmm(alpha, annotations)` / `AdditiveAttention.forward`의 masked energy와 `softmax(dim=-1)` / `AdditiveAttention.score`의 `v(tanh(W_s s + W_h h))` / 합성 monotonic alignment 정확도와 attention heatmap
- **Tensor shape 계약:** encoder/decoder [B,S/T,H] → attention [B,T,S] → context [B,T,H]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** context shape와 attention 행 합 1을 assert / padding 열의 확률이 정확히 0인지 검사 / energy shape 및 모든 파라미터 gradient 확인 / argmax alignment accuracy 및 padding mass로 정량 검증. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.Adam,Module.parameters,Optimizer.zero_grad,F.cross_entropy,Tensor.backward,Optimizer.step,Tensor.detach,torch.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 8개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
optimizer = torch.optim.Adam(attention.parameters(), lr=0.05)
loss_history = []
for _ in range(140):
    optimizer.zero_grad()
    _, _, energy = attention(decoder_queries, annotations, source_mask)
    loss = F.cross_entropy(energy, alignment_targets)
    loss.backward()
    optimizer.step()
    loss_history.append(float(loss.detach()))

assert loss_history[-1] < 0.2 * loss_history[0]
for parameter in attention.parameters():
    assert parameter.grad is not None and torch.isfinite(parameter.grad).all()

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** Figure 3: target 위치와 source annotation 사이 soft alignment
- **이 셀의 책임:** 합성 monotonic alignment 정확도와 attention heatmap
- **Tensor shape 계약:** encoder/decoder [B,S/T,H] → attention [B,T,S] → context [B,T,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** argmax alignment accuracy 및 padding mass로 정량 검증. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.inference_mode,Tensor.argmax -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

<details>
<summary><code>tensor.argmax(dim=None, keepdim=False)</code></summary>

#### `tensor.argmax(dim=None, keepdim=False)`

- **역할:** 지정 축에서 최댓값의 index를 찾습니다.
- **입력·반환:** 텐서와 축을 받고 정수 index 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit을 예측 class나 선택 action으로 바꿉니다.
- **주의점:** 미분 불가능하며 동점에서는 첫 index가 선택됩니다.

##### 관련 수식과 코드 연결

$$
k^\star=\operatorname*{arg\,max}_{k}x_k
$$

- **기호:** $k^\star$는 가장 큰 값 자체가 아니라 그 값의 index입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 어느 후보 집합에서 최댓값 index를 고를지 정합니다.
- **직관:** 연속 score를 이산 class·action 결정으로 바꾸는 선택 연산입니다.
- **shape 확인:** 선택 축이 제거되며 미분 가능한 gradient 경로는 만들지 않습니다.
- **공식 문서:** [Tensor.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
with torch.inference_mode():
    learned_context, learned_alpha, _ = attention(
        decoder_queries, annotations, source_mask
    )
predicted_alignment = learned_alpha.argmax(dim=1)
alignment_accuracy = (predicted_alignment == alignment_targets).float().mean().item()
padding_mass = learned_alpha[:, -1].sum().item()
identity_context = torch.eye(STEPS, device=DEVICE)
context_error = (learned_context[:, :STEPS] - identity_context).abs().mean().item()

assert alignment_accuracy >= 0.8
assert padding_mass == 0.0
assert context_error < 0.1
print(
    f"alignment_accuracy={alignment_accuracy:.1%}, "
    f"padding_mass={padding_mass:.1e}, "
    f"context_MAE={context_error:.4f}"
)

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** Figure 3: target 위치와 source annotation 사이 soft alignment
- **이 셀의 책임:** 합성 monotonic alignment 정확도와 attention heatmap
- **Tensor shape 계약:** encoder/decoder [B,S/T,H] → attention [B,T,S] → context [B,T,H]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** argmax alignment accuracy 및 padding mass로 정량 검증. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.3))
axes[0].plot(loss_history)
axes[0].set(title="alignment NLL", xlabel="step", ylabel="loss")
image = axes[1].imshow(
    learned_alpha.detach().cpu(),
    vmin=0,
    vmax=1,
    cmap="Blues",
    aspect="auto",
)
axes[1].set(
    title="soft alignment alpha(i,j)",
    xlabel="encoder position j (last = PAD)",
    ylabel="decoder step i",
    xticks=range(SOURCE_SLOTS),
    yticks=range(STEPS),
)
fig.colorbar(image, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

## 해석 체크

1. 왜 `softmax` 뒤가 아니라 앞에서 padding energy를 가려야 할까요?
2. context가 decoder step마다 달라지는 것이 고정 길이 encoder-decoder 병목을
   어떻게 완화하는지 설명하세요.

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `원 논문은 양방향 RNN encoder와 gated decoder를 WMT'14 영어-프랑스어 번역에 학습해 BLEU와 alignment를 평가했다. 여기서는 이미 만들어진 6차원 annotation과 5개 query만 사용하며 번역기 전체나 BLEU를 재현하지 않는다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.